# Import the necessary libraries

In [34]:
import pandas as pd
import statsmodels.api as sm
from statsmodels.tsa.api import VAR
from scipy import stats

# 1. Đọc dữ liệu
df = pd.read_csv('cleaned_data/VNM_macro_cleaned.csv')
df = df.sort_values('Year').reset_index(drop=True)

# 2. Lấy các biến số kinh tế vĩ mô cốt lõi và loại bỏ giá trị rỗng (NA)
cols = ['Year', 'GDP_Growth_Pct', 'Inflation_CPI_Pct', 'Unemployment_Pct', 'FDI_to_GDP_Pct', 'Economic_Openness_Pct']
df_clean = df[cols].dropna().copy()

print(f"Số lượng năm quan sát: {len(df_clean)}")

Số lượng năm quan sát: 35


# Giả thuyết 1: Tác động của FDI và Độ mở kinh tế lên Tăng trưởng GDP

In [35]:
# 3. Xây dựng mô hình OLS
# Khai báo các biến độc lập (X) và thêm Hằng số (Constant)
X = df_clean[['FDI_to_GDP_Pct', 'Economic_Openness_Pct']]
X = sm.add_constant(X)

# Khai báo biến phụ thuộc (y)
y = df_clean['GDP_Growth_Pct']

# Fit mô hình hồi quy tuyến tính (OLS)
model_gdp = sm.OLS(y, X).fit()

# 4. In bảng kết quả thống kê (Summary)
print("=== KẾT QUẢ MÔ HÌNH OLS (GIẢ THUYẾT 1) ===")
print(model_gdp.summary())

# 5. Kiểm định chẩn đoán phần dư (Jarque-Bera Test)
jb_stat, jb_p = stats.jarque_bera(model_gdp.resid)
print(f"\n=> Jarque-Bera p-value (Kiểm định phân phối chuẩn của phần dư): {jb_p:.4f}")
if jb_p > 0.05:
    print("Kết luận: Phần dư tuân theo phân phối chuẩn (Đạt yêu cầu).")
else:
    print("Kết luận: Phần dư KHÔNG tuân theo phân phối chuẩn (Vi phạm giả định).")

=== KẾT QUẢ MÔ HÌNH OLS (GIẢ THUYẾT 1) ===
                            OLS Regression Results                            
Dep. Variable:         GDP_Growth_Pct   R-squared:                       0.235
Model:                            OLS   Adj. R-squared:                  0.188
Method:                 Least Squares   F-statistic:                     4.924
Date:                Mon, 18 May 2026   Prob (F-statistic):             0.0137
Time:                        21:54:43   Log-Likelihood:                -59.890
No. Observations:                  35   AIC:                             125.8
Df Residuals:                      32   BIC:                             130.4
Df Model:                           2                                         
Covariance Type:            nonrobust                                         
                            coef    std err          t      P>|t|      [0.025      0.975]
-----------------------------------------------------------------------------

### ĐÁNH GIÁ VÀ GIẢI THÍCH KẾT QUẢ MÔ HÌNH OLS
**Giả thuyết 1:** *"Dòng vốn FDI và Độ mở thương mại thúc đẩy tăng trưởng kinh tế Việt Nam"*

Dựa trên kết quả hồi quy, phương trình mô hình được viết lại như sau:
$$\text{GDP\_Growth} = 6.8467 + 0.2449 \times \text{FDI\_to\_GDP} - 0.0121 \times \text{Economic\_Openness}$$

#### 1. Đánh giá độ phù hợp tổng thể của mô hình
- **Sức mạnh giải thích:** Mô hình giải thích được khoảng **23.5%** sự biến thiên của tốc độ tăng trưởng GDP ($R^2 = 0.235$; Adjusted $R^2 = 0.188$). 
- **Ý nghĩa tổng thể:** Kiểm định F có ý nghĩa thống kê ($Prob(F) = 0.0137 < 0.05$). Điều này khẳng định mô hình tổng thể có giá trị sử dụng và các biến độc lập được chọn có khả năng giải thích chung cho sự thay đổi của biến phụ thuộc.

#### 2. Đánh giá tác động của các hệ số hồi quy
- **Hằng số chặn (const):** Hệ số có giá trị dương ($6.8467$) và có ý nghĩa thống kê rất cao ($p < 0.001$). Về mặt kinh tế, điều này phản ánh mức tăng trưởng nền tảng nội tại. Nếu loại trừ tác động của FDI và độ mở thương mại, nền kinh tế Việt Nam vẫn duy trì động lực tăng trưởng ở mức xấp xỉ 6.85% nhờ các yếu tố nội sinh (tiêu dùng, đầu tư công,...).
- **Tỷ trọng FDI trên GDP (FDI_to_GDP_Pct):** Hệ số mang dấu dương ($+0.2449$) và có ý nghĩa thống kê ở mức 5% ($p = 0.043$). Cụ thể, khi tỷ trọng FDI/GDP tăng 1 điểm %, tốc độ tăng trưởng GDP tăng tương ứng khoảng **0.24 điểm %**. Kết quả này cung cấp bằng chứng thực nghiệm vững chắc chứng minh dòng vốn FDI đóng vai trò tích cực trong việc thúc đẩy tăng trưởng kinh tế.
- **Độ mở kinh tế (Economic_Openness_Pct):** Hệ số mang dấu âm ($-0.0121$) nhưng **không có ý nghĩa thống kê** ($p = 0.108 > 0.10$). Dựa trên tập dữ liệu này, chưa có đủ bằng chứng để khẳng định mức độ mở cửa thương mại có tác động trực tiếp lên tốc độ tăng trưởng GDP. Điều này có thể phản ánh đặc thù gia công xuất khẩu của Việt Nam: kim ngạch XNK lớn nhưng giá trị gia tăng nội địa giữ lại chưa cao.

#### 3. Chẩn đoán khuyết tật mô hình (Phân tích phần dư)
- **Kiểm định phân phối chuẩn:** Kiểm định Jarque-Bera cho $p-value = 0.520 > 0.05$. Không có bằng chứng bác bỏ giả thuyết $H_0$; phần dư của mô hình hoàn toàn tuân theo phân phối chuẩn, đảm bảo tính không chệch của các hệ số ước lượng.
- **Kiểm định tự tương quan:** Chỉ số Durbin-Watson đạt $1.640$, nằm trong vùng an toàn, cho thấy mô hình không gặp vấn đề tự tương quan bậc 1 nghiêm trọng.

> **KẾT LUẬN CHUNG:**
> Dữ liệu thực nghiệm **hỗ trợ một phần Giả thuyết 1**. Dòng vốn FDI có tác động thúc đẩy có ý nghĩa đối với tăng trưởng kinh tế Việt Nam, trong khi mức độ mở cửa thương mại chưa thể hiện được vai trò tác động thống kê một cách rõ ràng trong mô hình tuyến tính ngắn hạn.

# Giả thuyết 2: "Có sự đánh đổi giữa Lạm phát và Thất nghiệp tại Việt Nam" (Đường cong Phillips)

In [ ]:
# 7. Tạo biến lạm phát trễ (Inflation_Lag1)
df_clean['Inflation_Lag1'] = df_clean['Inflation_CPI_Pct'].shift(1)

# 8. Xóa bỏ dòng đầu tiên 
df_model = df_clean.dropna().copy()

# 9. Tạo lại biến giả Crisis_Dummy
crisis_years = [1990, 1997, 2008, 2011, 2020] # (Tùy chỉnh danh sách năm)
df_model['Crisis_Dummy'] = df_model['Year'].apply(lambda x: 1 if x in crisis_years else 0)

# 10. Chạy lại OLS với 3 biến: Thất nghiệp, Biến giả, và Lạm phát trễ
X_final = df_model[['Unemployment_Pct', 'Crisis_Dummy', 'Inflation_Lag1']]
X_final = sm.add_constant(X_final)
y_final = df_model['Inflation_CPI_Pct']

model_final = sm.OLS(y_final, X_final).fit()

# In kết quả
print(model_final.summary())

                            OLS Regression Results                            
Dep. Variable:      Inflation_CPI_Pct   R-squared:                       0.487
Model:                            OLS   Adj. R-squared:                  0.436
Method:                 Least Squares   F-statistic:                     9.488
Date:                Mon, 18 May 2026   Prob (F-statistic):           0.000145
Time:                        21:58:14   Log-Likelihood:                -88.906
No. Observations:                  34   AIC:                             185.8
Df Residuals:                      30   BIC:                             191.9
Df Model:                           3                                         
Covariance Type:            nonrobust                                         
                       coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------------
const                8.5723      3.237  

### ĐÁNH GIÁ VÀ GIẢI THÍCH KẾT QUẢ MÔ HÌNH OLS
**Giả thuyết 2:** *"Có sự đánh đổi giữa Lạm phát và Thất nghiệp tại Việt Nam (Đường cong Phillips) và tác động của các cú sốc."*

Dựa trên kết quả hồi quy, phương trình mô hình được viết lại như sau:
$$\text{Inflation} = 8.5723 - 2.9012 \times \text{Unemployment} + 7.2610 \times \text{Crisis\_Dummy} + 0.2958 \times \text{Inflation\_Lag1}$$

#### 1. Đánh giá độ phù hợp tổng thể của mô hình
- **Sức mạnh giải thích:** Mô hình giải thích được khoảng **48.7%** sự biến thiên của tỷ lệ lạm phát ($R^2 = 0.487$; Adjusted $R^2 = 0.436$). Đây là một mức độ giải thích khá tốt đối với dữ liệu chuỗi thời gian vĩ mô nhiều biến động như lạm phát.
- **Ý nghĩa tổng thể:** Kiểm định F có ý nghĩa thống kê rất cao ($Prob(F) = 0.000145 < 0.01$). Điều này khẳng định mô hình tổng thể hoàn toàn phù hợp và các biến độc lập kết hợp lại có khả năng giải thích hiệu quả sự thay đổi của lạm phát.

#### 2. Đánh giá tác động của các hệ số hồi quy
- **Tỷ lệ thất nghiệp (Unemployment_Pct):** Hệ số mang dấu âm ($-2.9012$) và có ý nghĩa thống kê ở mức 10% ($p = 0.059 < 0.10$). Điều này củng cố sự tồn tại (dù khá yếu) của đường cong Phillips tại Việt Nam: khi thất nghiệp tăng 1 điểm %, lạm phát có xu hướng giảm khoảng 2.9 điểm %. Tuy nhiên, vì mức ý nghĩa chỉ đạt 10%, sức ép từ thị trường lao động không phải là yếu tố chi phối mạnh nhất đến giá cả.
- **Biến giả khủng hoảng (Crisis_Dummy):** Hệ số mang dấu dương ($+7.2610$) và có ý nghĩa thống kê cực kỳ cao ($p = 0.001$). Trong các năm xảy ra cú sốc lạm phát phi mã, có một yếu tố ngoại lai (ví dụ: đứt gãy chuỗi cung ứng, giá dầu thế giới tăng) đã trực tiếp đẩy lạm phát của Việt Nam cao hơn bình thường **7.26 điểm %**, hoàn toàn độc lập với các yếu tố nội tại.
- **Lạm phát trễ 1 năm (Inflation_Lag1):** Hệ số dương ($+0.2958$) và có ý nghĩa thống kê ở mức 5% ($p = 0.046$). Con số này minh chứng cho **"tính quán tính" (độ ỳ)** của lạm phát: mức lạm phát của năm trước cứ tăng 1 điểm % thì sẽ tự động truyền sang năm nay khoảng 0.3 điểm %. Điều này thường do tâm lý kỳ vọng giá tăng của người dân hoặc độ trễ trong việc điều chỉnh các hợp đồng kinh tế.
- **Hằng số chặn (const):** Có giá trị $8.5723$ ($p = 0.013$). Về mặt toán học, đây là mức lạm phát nền; tuy nhiên trong thực tế, nó đóng vai trò là mỏ neo định vị đường hồi quy nhiều hơn là một ý nghĩa kinh tế độc lập (vì thất nghiệp không bao giờ bằng 0).

#### 3. Chẩn đoán khuyết tật mô hình (Phân tích phần dư)
- **Kiểm định phân phối chuẩn (Jarque-Bera):** Chỉ số JB có $p-value = 0.165 > 0.05$. Do đó, mô hình không vi phạm giả định phân phối chuẩn. Việc tách các năm khủng hoảng ra bằng biến giả đã xử lý triệt để hiện tượng nhiễu ngoại lai (outliers), giúp phần dư trở nên đối xứng và ổn định.
- **Kiểm định tự tương quan (Durbin-Watson):** Chỉ số DW đạt **1.727**, tiến khá sát về mức lý tưởng 2.0. Việc bổ sung biến lạm phát trễ (`Inflation_Lag1`) đã triệt tiêu thành công hiện tượng tự tương quan bậc 1, đảm bảo tính không chệch và tin cậy của các hệ số (p-value).

> **KẾT LUẬN CHUNG:**
> Lạm phát tại Việt Nam chịu ảnh hưởng chủ yếu từ các **cú sốc ngoại lai (nhập khẩu lạm phát)** và **tính quán tính từ quá khứ**, hơn là sự đánh đổi mạnh mẽ với tỷ lệ việc làm nội địa. Mô hình hiện tại đã tuân thủ tốt các giả định khắt khe của OLS, đem lại kết quả ước lượng đáng tin cậy.